In [1]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
import torch

In [2]:
from peft import LoraConfig, get_peft_model

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
model_name = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
data_path = '/content/drive/MyDrive/llm_finetuning/data/data.jsonl'
data = load_dataset("json",data_files=data_path)

Generating train split: 0 examples [00:00, ? examples/s]

In [6]:
config = LoraConfig(
    r=4,
    lora_alpha=8,
    target_modules=["c_attn"],
    lora_dropout=0.1,
    bias="none"
)

model = get_peft_model(model, config)

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2285: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


In [32]:
def format_example(example):
    text = f"Input: {example['input']}\nOutput: {example['output']}"

    tokenized = tokenizer(
        text,
        truncation=True,
        padding="max_length",
        max_length=128
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = data.map(format_example)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [33]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [34]:
training_args = TrainingArguments(
    output_dir='/content/drive/MyDrive/llm_finetuning/gpt2',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_train_epochs=15,   # overfit intentionally
    logging_steps=5,
    save_steps=20,
    fp16=False,  # CPU safe
    report_to="none"
)

In [35]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    data_collator=data_collator
)

trainer.train()

Step,Training Loss
5,5.354712
10,5.386977
15,5.308436
20,5.336277
25,5.246793
30,5.317655
35,5.278365
40,5.228927
45,5.294947
50,5.167664


TrainOutput(global_step=75, training_loss=5.263630091349284, metrics={'train_runtime': 8.1724, 'train_samples_per_second': 36.709, 'train_steps_per_second': 9.177, 'total_flos': 9815615078400.0, 'train_loss': 5.263630091349284, 'epoch': 15.0})

In [36]:
save_path = '/content/drive/MyDrive/llm_finetuning/gpt2'
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

('/content/drive/MyDrive/llm_finetuning/gpt2/tokenizer_config.json',
 '/content/drive/MyDrive/llm_finetuning/gpt2/tokenizer.json')

In [ ]:
### Prediction on data

In [37]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_path = '/content/drive/MyDrive/llm_finetuning/gpt2'

saved_tokenizer = AutoTokenizer.from_pretrained(model_path)
saved_model = AutoModelForCausalLM.from_pretrained(model_path)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights: 0it [00:00, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /content/drive/MyDrive/llm_finetuning/gpt2
Key                                                                                 | Status     | 
------------------------------------------------------------------------------------+------------+-
base_model.model.transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_B.default.weight | UNEXPECTED | 
base_model.model.transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_A.default.weight | UNEXPECTED | 
transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_B.default.weight                  | MISSING    | 
transformer.h.{0, 1, 2, 3, 4, 5}.attn.c_attn.lora_A.default.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [40]:
def predict_intent(user_input):
    prompt = f"Input: {user_input}\nOutput:"

    inputs = tokenizer(prompt, return_tensors="pt")

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result

In [41]:
user_input = "I want to know my account balance"
predict_intent(user_input)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


'Input: I want to know my account balance\nOutput:\nI want'

In [46]:
def predict_intent(user_input):
    prompt = f"""
              Input: Show me my balance
              Output: check_balance

              Input: Pay my bill
              Output: bill_payment

              Input: What is my due amount
              Output: due_amount

              Input: {user_input}
              Output:
              """

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=3,
        do_sample=False
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return result

In [48]:
user_input = "I want to know my account balance"
print(predict_intent(user_input))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



              Input: Show me my balance
              Output: check_balance

              Input: Pay my bill
              Output: bill_payment

              Input: What is my due amount
              Output: due_amount

              Input: I want to know my account balance
              Output:
                 


In [53]:
def predict_intent(user_input):
    prompt = f"""Input: Show me my balance
              Output: check_balance

              Input: Pay my bill
              Output: bill_payment

              Input: What is my due amount
              Output: due_amount

              Input: {user_input}
              Output:"""

    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=True
    )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print("RAW:", result)  # debug

    return result.split("Output:")[-1].strip()

In [54]:
user_input = "I want to know my due payment amount"
print(predict_intent(user_input))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


RAW: Input: Show me my balance
              Output: check_balance

              Input: Pay my bill
              Output: bill_payment

              Input: What is my due amount
              Output: due_amount

              Input: I want to know my due payment amount
              Output: onpayment_me : I want to know my
onpayment_me : I want to know my
